# nb53 - Low-energy error analysis (toward aggregate 0.030)

**Error analysis stage (this notebook IS the stage).** Goal reset 2026-08-01: aggregate 0.030. Quantile math makes the lowest bin the largest lever: bin 1 (2.2-10.7 GeV) sits at 0.0629 vs its floor 0.039 - a 0.024 gap in the single most populated region of the spectrum, larger than all E>17 gaps combined. The whole campaign so far attacked E>17; low E was never decomposed.

**Question.** Which error axis dominates the low-E residuals of the 0.0409 stack: pileup contamination (fractionally largest at low E), containment, region/pitch, seed misidentification, or timing coverage?

**Method.** Inference-only decomposition of the saved nb52 stack prediction on the fixed test split: per-axis tertile widths and correlations, low-E only (E < 10.7 GeV), stratified against the same axes at mid/high E as control. No training, no tuning; the next hypothesis (H17) is whatever axis wins.

In [1]:
import os, sys, time, pathlib
import numpy as np, pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
pio.templates.default = 'plotly_white'
REPO = pathlib.Path(os.environ['REPO_DIR']) if os.environ.get('REPO_DIR') else (
    pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd())
sys.path.insert(0, str(REPO / 'scripts'))
from run_experiments import resolution, PITCH, EPS
from picocal_data import build_grid, make_windows, splits_for, prep, THRESH
OUT = REPO / 'reports' / 'predictions'
t0 = time.time()
ME = build_grid(sorted((REPO / 'data' / 'minimum_bias').glob('*.root')), 'minbias')
CE = build_grid(sorted((REPO / 'data' / 'full').glob('matched_*.root')), 'clean')
D = prep(4, ME, CE, ng=6)
kte = D['kte']
pe = np.load(OUT / 'nb52_pred_stack.npy')
te = D['Et'][kte]
r = (pe - te) / te
print(f'stack check: {resolution(pe, te)["sigma_eff"]:.4f} (expect 0.0409) | build {time.time()-t0:.0f}s')

minbias: 72554 events


clean: 30303 events


W=4: N 102857 (main 72554 + aux 30303), tr/va/te 50787/10883/10884, IN_DIM 16
stack check: 0.0409 (expect 0.0409) | build 135s


In [2]:
rows, keep = make_windows(4, ME)
ktr_m, kva_m, kte_m = splits_for(keep, len(ME))
sumE = np.array([rw[1] for rw in rows], np.float32)
seedE = np.array([rw[2] for rw in rows], np.float32)
reg = np.array([rw[4] for rw in rows], int)
ncell = np.array([rw[0].shape[0] for rw in rows], np.float32)
tfrac = np.array([float(np.mean(rw[0][:, 9] + rw[0][:, 10] > 0)) for rw in rows], np.float32)
n_mb = len(rows)
kte_pos = np.asarray(kte)
axes = dict(
    contamination=(sumE / np.maximum(1000.0 * D['Et'][:n_mb], EPS))[kte_pos],
    seed_fraction=(seedE / np.maximum(sumE, EPS))[kte_pos],
    region=reg[kte_pos].astype(float),
    n_cells=ncell[kte_pos],
    timed_fraction=tfrac[kte_pos])
lo = te < 10.7
mid = (te >= 10.7) & (te < 24.0)
print(f'low-E events: {lo.sum()} | sigma_eff low {resolution(pe[lo], te[lo])["sigma_eff"]:.4f} | mid {resolution(pe[mid], te[mid])["sigma_eff"]:.4f}')

low-E events: 1793 | sigma_eff low 0.0626 | mid 0.0401


Per-axis decomposition at low E: sigma_eff in tertiles of each axis (region uses its discrete values), plus the same split at mid E as control. The winning axis is the one with the largest low-E spread that is NOT mirrored at mid E (a mirrored spread is just the axis correlating with energy itself).

In [3]:
def tertile_widths(x, mask):
    xm = x[mask]; pm = pe[mask]; tm = te[mask]
    qs = np.quantile(xm, [1/3, 2/3])
    out = []
    for gsel, lab in [(xm < qs[0], 'lo'), ((xm >= qs[0]) & (xm < qs[1]), 'mid'), (xm >= qs[1], 'hi')]:
        if gsel.sum() < 80: out.append((lab, np.nan, 0)); continue
        out.append((lab, resolution(pm[gsel], tm[gsel])['sigma_eff'], int(gsel.sum())))
    return out
def region_widths(mask):
    out = []
    for g in sorted(set(axes['region'][mask].astype(int))):
        gsel = mask & (axes['region'] == g)
        if gsel.sum() < 80: continue
        out.append((f'{int(PITCH[g])}mm', resolution(pe[gsel], te[gsel])['sigma_eff'], int(gsel.sum())))
    return out
print(f'{"axis":18s} {"low-E tertiles (lo/mid/hi)":42s} {"mid-E control"}')
summary = {}
for name, x in axes.items():
    if name == 'region':
        lw = region_widths(lo); mw = region_widths(mid)
        fmt = lambda ws: ' '.join(f'{l}:{s:.4f}' for l, s, n in ws)
        print(f'{name:18s} {fmt(lw):42s} {fmt(mw)}')
        summary[name] = (max(s for _, s, _ in lw) - min(s for _, s, _ in lw)) if lw else 0
    else:
        lw = tertile_widths(x, lo); mw = tertile_widths(x, mid)
        fmt = lambda ws: ' / '.join(f'{s:.4f}' if np.isfinite(s) else 'n/a' for _, s, _ in ws)
        print(f'{name:18s} {fmt(lw):42s} {fmt(mw)}')
        vals = [s for _, s, _ in lw if np.isfinite(s)]
        mvals = [s for _, s, _ in mw if np.isfinite(s)]
        summary[name] = (max(vals) - min(vals)) - 0.5 * (max(mvals) - min(mvals)) if vals and mvals else 0
print()
for name, sc in sorted(summary.items(), key=lambda kv: -kv[1]):
    print(f'excess low-E spread {name:18s} {sc:+.4f}')

axis               low-E tertiles (lo/mid/hi)                 mid-E control
contamination      0.0415 / 0.0548 / 0.1070                   0.0264 / 0.0333 / 0.0772
seed_fraction      0.0976 / 0.0585 / 0.0433                   0.0680 / 0.0357 / 0.0278
region             40mm:0.0807 60mm:0.0533 120mm:0.0528       15mm:0.2874 30mm:0.1106 40mm:0.0480 60mm:0.0299 120mm:0.0316
n_cells            0.0599 / 0.0518 / 0.0766                   0.0320 / 0.0341 / 0.0584
timed_fraction     0.0901 / 0.0532 / 0.0513                   0.0717 / 0.0349 / 0.0287

excess low-E spread contamination      +0.0401
excess low-E spread seed_fraction      +0.0342
excess low-E spread region             +0.0279
excess low-E spread timed_fraction     +0.0173
excess low-E spread n_cells            +0.0116


Bias structure at low E: median residual vs energy (is the width actually a bias ladder?) and the residual distribution shape per contamination tertile.

In [4]:
fig = go.Figure()
ebins = np.linspace(te[lo].min(), 10.7, 9)
ctr, med, w68 = [], [], []
for i in range(8):
    mm = lo & (te >= ebins[i]) & (te < ebins[i+1])
    if mm.sum() < 80: continue
    ctr.append(float(np.median(te[mm])))
    med.append(float(np.median(r[mm])))
    w68.append(resolution(pe[mm], te[mm])['sigma_eff'])
fig.add_scatter(x=ctr, y=med, mode='lines+markers', name='median bias')
fig.add_scatter(x=ctr, y=w68, mode='lines+markers', name='sigma_eff')
fig.update_layout(height=400, title='low-E: bias and width vs energy',
                  xaxis_title='true energy [GeV]', yaxis_title='relative')
fig.write_html(REPO / 'reports' / 'figures' / 'interactive' / 'ifig9_lowE_bias.html', include_plotlyjs='cdn')
fig.show()
cont = axes['contamination']
qs = np.quantile(cont[lo], [1/3, 2/3])
fig2 = go.Figure()
for gsel, lab in [(cont < qs[0], 'low contamination'), ((cont >= qs[0]) & (cont < qs[1]), 'mid'), (cont >= qs[1], 'high contamination')]:
    mm = lo & gsel
    fig2.add_histogram(x=np.clip(r[mm], -0.4, 0.4), nbinsx=80, name=f'{lab} (n={mm.sum()})', opacity=0.55)
fig2.update_layout(barmode='overlay', height=400, title='low-E residuals by contamination tertile',
                   xaxis_title='(Epred-Etrue)/Etrue', yaxis_title='events')
fig2.write_html(REPO / 'reports' / 'figures' / 'interactive' / 'ifig10_lowE_contamination.html', include_plotlyjs='cdn')
fig2.show()

## Findings

**Contamination dominates low E, and by more than anywhere else.** Excess low-E spread (tertile spread minus half the mid-E control spread): contamination +0.040, seed_fraction +0.034, region +0.028, timed_fraction +0.017, n_cells +0.012. The three leading axes are one phenomenon seen three ways - pileup energy in the window (seed_fraction and timed_fraction are contamination proxies: pileup lowers the seed's share and pileup cells lack times).

**The clean-contamination tertile already sits at 0.0415 vs the ~0.039 bin floor** - low-E events without heavy pileup are nearly solved. The entire 0.063 -> 0.039 gap lives in the contaminated two-thirds (mid 0.0548, high 0.1070).

**H17 verdict: this is a supervision problem, not a modeling problem.** Every label-free contamination mechanism was measured and falsified this campaign (H2-H16); the axis that dominates low E is exactly the one the minbias-only positional-matching sample (Felipe, incoming) addresses. Quantified stake for the 2026-08-13 meeting: real per-cell pileup labels target a ~0.02 reduction in the most populated bin - the single largest remaining contribution toward aggregate 0.030.